In [ ]:
import os
import polars as pl
import numpy as np
import random

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

import matplotlib.pyplot as plt

random.seed(4)
np.random.seed(4)
torch.manual_seed(4)

In [ ]:
class AveragePool(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x, masks=None):
        B, N, D = x.shape

        if masks is not None:
            return x.sum(dim=1) / masks.float().sum(dim=1).unsqueeze(1)
        return x.mean(dim=1)
    
class GatedPool(nn.Module):
    def __init__(self, embed_dim: int):
        super().__init__()
        self.gate_mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.Sigmoid()
        )

    def forward(self, x, mask=None):
        B, N, D = x.shape
        gate_weights = self.gate_mlp(x)
        
        gated_features = x * gate_weights

        if mask is not None:
            gated_features = gated_features * mask.unsqueeze(-1)

        return torch.sum(gated_features, dim=1)
    
class SoftAttentionPool(nn.Module):
    def __init__(self, embed_dim: int):
        super().__init__()
        self.attention_weights_mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.ReLU(),
            nn.Linear(embed_dim // 2, 1)
        )

    def forward(self, x, mask=None):
        B, N, D = x.shape
        raw_scores = self.attention_weights_mlp(x)

        if mask is not None:
            raw_scores = raw_scores.masked_fill(~mask.unsqueeze(-1), float('-inf'))
        
        attention_weights = torch.softmax(raw_scores, dim=1)

        weighted_embeddings = x * attention_weights

        return torch.sum(weighted_embeddings, dim=1)

class AttentionPool(nn.Module):
    def __init__(self, embed_dim: int):
        super().__init__()
        self.query = nn.Parameter(torch.zeros(1, embed_dim))
        
        self.attention_net = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.Tanh(),
            nn.Linear(embed_dim, 1)
        )
        
        nn.init.xavier_uniform_(self.query)

    def forward(self, x, mask=None):
        B, N, D = x.shape
        
        query_expanded = self.query.expand(B, N, -1)
        
        combined = x + query_expanded

        raw_scores = self.attention_net(combined)

        if mask is not None:
            raw_scores = raw_scores.masked_fill(~mask.unsqueeze(-1), float('-inf'))
        
        attention_weights = torch.softmax(raw_scores, dim=1)
        weighted_sum = torch.sum(x * attention_weights, dim=1)

        return weighted_sum

In [ ]:
class Classifier(nn.Module):
    def __init__(self, pooler, embed_dim: int, num_classes: int):
        super().__init__()
        self.pooler = pooler
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x, mask=None):
        x = self.pooler(x, mask)
        logits = self.fc(x)
        return logits
    
class Regressor(nn.Module):
    def __init__(self, pooler, embed_dim: int):
        super().__init__()
        self.pooler = pooler
        self.fc = nn.Linear(embed_dim, 1)

    def forward(self, x, mask=None):
        x = self.pooler(x, mask)
        prediction = self.fc(x)
        return prediction

In [ ]:
def add_embedding_noise(embeddings, sigma=0.1, p=1.0):
    """
    embeddings: torch.Tensor of shape (N, D)
    sigma: standard deviation of Gaussian noise
    p: probability of applying noise
    """
    if torch.rand(1).item() < p:
        noise = torch.randn_like(embeddings) * sigma
        embeddings = embeddings + noise
    return embeddings

class EmbeddingDataset(Dataset):
    def __init__(self, patient_ids, id_to_path, labels, train=True):
        self.patient_ids = patient_ids
        self.id_to_path = id_to_path
        self.labels = labels
        self.train = train

    def __len__(self):
        return len(self.patient_ids)
    
    def get_labels(self):
        return [self.labels[x] for x in self.patient_ids]

    def __getitem__(self, idx):
        patient_id = self.patient_ids[idx]
        embedding_path = self.id_to_path[patient_id]
        
        embedding_data = torch.load(embedding_path)
        embeddings = embedding_data["cls"]
        label = float(self.labels[patient_id])

        if self.train:
            embeddings = add_embedding_noise(embeddings)

        return embeddings, label

In [ ]:
def collate_fn_regression(batch):
    batch.sort(key=lambda x: x[0].shape[0], reverse=True)
    
    embeddings_list, labels_list = zip(*batch)
    
    max_len = embeddings_list[0].shape[0]
    padded_embeddings = torch.zeros(len(embeddings_list), max_len, embeddings_list[0].shape[1])
    masks = torch.zeros(len(embeddings_list), max_len, dtype=torch.bool)
    
    for i, embedding in enumerate(embeddings_list):
        seq_len = embedding.shape[0]
        padded_embeddings[i, :seq_len, :] = embedding
        masks[i, :seq_len] = True
        
    labels = torch.tensor(labels_list, dtype=torch.float32).unsqueeze(1)
    
    return padded_embeddings, labels, masks

def collate_fn_class(batch):
    batch.sort(key=lambda x: x[0].shape[0], reverse=True)
    
    embeddings_list, labels_list = zip(*batch)
    
    max_len = embeddings_list[0].shape[0]
    padded_embeddings = torch.zeros(len(embeddings_list), max_len, embeddings_list[0].shape[1])
    masks = torch.zeros(len(embeddings_list), max_len, dtype=torch.bool)
    
    for i, embedding in enumerate(embeddings_list):
        seq_len = embedding.shape[0]
        padded_embeddings[i, :seq_len, :] = embedding
        masks[i, :seq_len] = True
        
    labels = torch.tensor(labels_list, dtype=torch.long)
    
    return padded_embeddings, labels, masks

In [ ]:
embeddings_path = "/scratch/VM/radio-foundation/cache/embeddings/NSCLC_Radiomics"

files = [x for x in os.listdir(embeddings_path) if x.endswith(".pth")]
id_to_path = {
    f.replace(".pth", "") : os.path.join(embeddings_path, f) for f in files
}

In [ ]:
data_path = "/scratch/VM/radio-foundation/datasets"

clinical_raw = pl.read_csv(os.path.join(data_path, "NSCLC-Radiomics/clinical.csv"))
clinical_raw.head()

In [ ]:
label_name = "clinical.T.Stage"
clinical_raw[label_name].value_counts()

In [ ]:
df = clinical_raw.filter(pl.col(label_name) != 'NA')
labels = {
    key: int(value) - 1
    for key, value in zip(df['PatientID'], df[label_name])
    if value in ['1', '2', '3','4']
}
num_classes = 4

In [ ]:
patient_ids = list(labels.keys())
exist_patient_ids = [x for x in patient_ids if x in id_to_path.keys()]
exist_labels = [labels[pid] for pid in exist_patient_ids]
train_ids, val_ids = train_test_split(exist_patient_ids, test_size=0.2, random_state=5, stratify=exist_labels)

train_dataset = EmbeddingDataset(train_ids, id_to_path, labels, train=True)
val_dataset = EmbeddingDataset(val_ids, id_to_path, labels, train=False)

In [ ]:
class_weights = [len(exist_labels) / num_classes / exist_labels.count(x) for x in range(num_classes)]
class_weights

In [ ]:
def do_train(
        model,
        optimizer,
        loss_fn,
        train_dataloader,
        val_dataloader,
        num_epochs,
        device,
        verbose=False
    ):
    train_loss_list = []
    val_loss_list = []

    best_val_loss = float("inf")
    best_model_state = model.state_dict()
    
    for epoch in range(num_epochs):

        model.train()
        train_loss = 0.0
        for embeddings, labels, masks in train_dataloader:
            embeddings, labels, masks = embeddings.to(device), labels.to(device), masks.to(device)
            
            predictions = model(embeddings, masks)
            loss = loss_fn(predictions, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * embeddings.size(0)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for embeddings, labels, masks in val_dataloader:
                embeddings, labels, masks = embeddings.to(device), labels.to(device), masks.to(device)
                
                predictions = model(embeddings, masks)
                loss = loss_fn(predictions, labels)
                
                val_loss += loss.item() * embeddings.size(0)

        avg_train_loss = (train_loss / len(train_dataset))
        avg_val_loss = (val_loss / len(val_dataset))

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = model.state_dict()

        train_loss_list.append(avg_train_loss)
        val_loss_list.append(avg_val_loss)
        
        if epoch % 10 == 0 and verbose:
            print(f"Epoch [{epoch+1}/{num_epochs}], "
                    f"Training MSE: {avg_train_loss:.4f}, "
                    f"Validation MSE: {avg_val_loss:.4f}")
            
    return train_loss_list, val_loss_list, best_model_state

In [ ]:
def get_predictions(model, dataloader, device):
    all_labels = []
    all_predictions = []
    model.eval()
    with torch.no_grad():
        for embeddings, labels, masks in dataloader:
            embeddings, labels, masks = embeddings.to(device), labels.to(device), masks.to(device)
            
            predictions = model(embeddings, masks)
            all_labels.append(labels.cpu())
            all_predictions.append(predictions.cpu())

    all_predictions = torch.cat(all_predictions, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    return all_labels, all_predictions

In [ ]:
EMBED_DIM = 768
num_epochs = 50
batch_size = 16
learning_rate = 0.0001

train_dataloader = DataLoader(
    train_dataset,
    shuffle=True,
    collate_fn=collate_fn_class,
    batch_size=batch_size,
)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_class)

device = torch.device("cuda")
pooler = AttentionPool(EMBED_DIM)
#model = Regressor(pooler, embed_dim=EMBED_DIM).to(device)
model = Classifier(pooler, embed_dim=EMBED_DIM, num_classes=4).to(device)

#loss_fn = nn.MSELoss()
loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(class_weights).to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

train_loss_list, val_loss_list, best_model_state = do_train(
    model,
    optimizer,
    loss_fn,
    train_dataloader,
    val_dataloader,
    num_epochs,
    device
)
model.load_state_dict(best_model_state)


In [ ]:
val_vmin = min(val_loss_list)

plt.axhline(y=val_vmin, color='k', linestyle=':', alpha=0.5)
plt.plot(train_loss_list, label="Train")
plt.plot(val_loss_list, label="Validation")
plt.grid(True)
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.show()

In [ ]:
all_labels, all_predictions = get_predictions(model, val_dataloader, device)
plt.scatter(all_labels, all_predictions)
plt.xlabel("True Labels")
plt.ylabel("Predicted Labels")
#plt.xticks([1, 2, 3, 4])
#plt.yticks([1, 2, 3, 4])
plt.grid(True)
plt.show()

In [ ]:
all_labels, all_predictions = get_predictions(model, val_dataloader, device)
all_predictions = torch.nn.functional.softmax(all_predictions, dim=1)
all_predictions = torch.argmax(all_predictions, dim=1)

cm = confusion_matrix(all_labels, all_predictions)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")

plt.colorbar(im, ax=ax)
ax.set(
    xlabel="Predicted label",
    ylabel="True label",
    title="Confusion Matrix"
)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j],
                ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black")
        
ax.set_xticks(range(cm.shape[0]))
ax.set_yticks(range(cm.shape[0]))

plt.tight_layout()
plt.show()